# Packages import

In [8]:
import os
import json
import requests
from bs4 import BeautifulSoup

: 

# Ceneo scraper

1. Provide url address of product's opinions webpage

In [2]:
product_code = "34935197"
page = 1
url = f"https://www.ceneo.pl/{product_code}/opinie-{page}"

2. Send the request to provide url address

In [ ]:
response = requests.get(url)
print(response.status_code)

3. If status code is ok, fetch product name

In [ ]:
page_dom = BeautifulSoup(response.text, 'html.parser')
print(type(page_dom))

In [ ]:
product_name = page_dom.select_one("h1.product-top__product-info__name").get_text(strip=True)
print(product_name)

4. If status code is ok, fetc all opinions from requested webpage

In [ ]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

In [ ]:
all_opinions = page_dom.find_all("div", {"class": "js_product-review:not(.user-post--highlight)"})
opinions = [r for r in all_opinions if "user-post--highlight" not in r.get('clas', [])]

5. For all fetched opnions, parse them to extract relevant data

In [ ]:
all_opinions = []
for opinion in opinions:
    single_opinion = {
        'opinion_id': opinion['data-entry-id'],
        'author': opinion.select_one('span.user-post__author-name').get_text().strip(),
        'recommendation': opinion.select_one('span.user-post__author-recommendation > em').get_text().strip() if opinion.select_one('span.user-post__author-recommendation > em') else None,
        'score': opinion.select_one('span.user-post__score-count').get_text().strip(),
        'content': opinion.select_one('	div.user-post__text').get_text().strip(),
        'pros': [p.get_text() for p in opinion.select('div.review-feature__item--positive')],
        'cons': [c.get_text() for c in opinion.select('div.review-feature__item--negative')],
        'like': opinion.select_one('button.vote-yes > span').get_text().strip(),
        'dislike': opinion.select_one('button.vote-no > span').get_text().strip(),
        'publish_date': opinion.select_one('span.user-post__published > time:nth-child(1)[datetime]').['datetime'].strip(),
        'purchase_date': opinion.select_one('span.user-post__published > time:nth-child(2)[datetime]').['datetime'].strip() if opinion.select_one('span.user-post__published > time:nth-child(2)[datetime]') else None,
    }
    all_opinions.append(single_opinion)

: 

6. Check if there is next page with opinions

In [ ]:
next = True if page_dom.select_one('button.pagination__next') else False
if next: page += 1

8. Save obtained opinions

In [ ]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [ ]:
with open(f"./{product_code}.json", "w", encoding="UTF-8") as jf:
    json.dump(all_opinions, jf, indent=4, ensure_ascii=False)